# Fruit detection — two-stage training on Colab

Pipeline mới gồm detector `mango/dragonfruit`, classifier xoài 4 mức và classifier thanh long 3 mức. Bật **T4 GPU** trước khi chạy. Mỗi phần train nằm ở một cell riêng để có thể chạy/resume độc lập.


In [ ]:
# STEP 1 — Clone code and install dependencies
import os, subprocess, sys
REPO_URL = 'https://github.com/khuyenabc123/fruit_detection.git'
REPO = '/content/fruit_detection'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/training/requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Hãy bật Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))


## STEP 2 — Mount Drive và giải nén hai bộ Roboflow cũ

Đặt `mango_dataset.zip` và `dragonfruit_dataset.zip` trong `MyDrive/fruit_data/`. Hai bộ này cung cấp bounding box thanh long và nhãn độ chín; dữ liệu public mới không đủ để thay thế chúng hoàn toàn.


In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
drive.mount('/content/drive')

def extract_rf(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    assert zip_path.is_file(), f'Missing {zip_path}'
    destination.mkdir(parents=True, exist_ok=True)
    if not list(destination.rglob('data.yaml')):
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(destination)
    yaml_files = list(destination.rglob('data.yaml'))
    assert len(yaml_files) == 1, f'Expected one data.yaml under {destination}, found {yaml_files}'
    return yaml_files[0].parent

MANGO_RF = extract_rf('/content/drive/MyDrive/fruit_data/mango_dataset.zip', '/content/rf/mango')
DRAGON_RF = extract_rf('/content/drive/MyDrive/fruit_data/dragonfruit_dataset.zip', '/content/rf/dragon')
print('Mango:', MANGO_RF)
print('Dragon:', DRAGON_RF)


In [ ]:
# STEP 3 — Cache public archives on Drive; extract to fast local Colab storage
public_download_env = os.environ.copy()
public_download_env['PUBLIC_DATA_RAW_DIR'] = '/content/drive/MyDrive/fruit_data/public_raw'
public_download_env['PUBLIC_DATA_EXTRACTED_DIR'] = f'{REPO}/data/external/extracted'
subprocess.run(
    ['bash', f'{REPO}/scripts/download_public_datasets.sh', 'two-stage-minimal'],
    cwd=REPO, env=public_download_env, check=True,
)


In [ ]:
# STEP 4 — Prepare leak-resistant detector and classifier datasets
DATASET_ROOT = Path('/content/two_stage_dataset')
prepare_command = [
    sys.executable, f'{REPO}/training/prepare_two_stage_data.py',
    '--external-root', f'{REPO}/data/external/extracted',
    '--mango-rf', str(MANGO_RF),
    '--dragon-rf', str(DRAGON_RF),
    '--output', str(DATASET_ROOT),
    '--mode', 'copy', '--overwrite', '--seed', '42',
]
subprocess.run(prepare_command, check=True)


In [ ]:
# STEP 5 — Mandatory audit gate; do not train if this cell fails
import json, shutil
sys.path.insert(0, f'{REPO}/training')
from train_two_stage import validate_detector_dataset, validate_classifier_dataset
validate_detector_dataset(DATASET_ROOT / 'detector/data.yaml')
validate_classifier_dataset(DATASET_ROOT / 'mango_classifier', 'mango')
validate_classifier_dataset(DATASET_ROOT / 'dragon_classifier', 'dragon')
report = json.loads((DATASET_ROOT / 'preparation_report.json').read_text())
print(json.dumps(report, indent=2, ensure_ascii=False))

# Preserve the exact split manifests on Drive for reproducibility.
manifest_dir = Path('/content/drive/MyDrive/fruit_two_stage/manifests')
manifest_dir.mkdir(parents=True, exist_ok=True)
for path in DATASET_ROOT.rglob('manifest.csv'):
    shutil.copy2(path, manifest_dir / f'{path.parent.name}_manifest.csv')
shutil.copy2(DATASET_ROOT / 'preparation_report.json', manifest_dir / 'preparation_report.json')


## STEP 6A — Train/resume detector
Detector dùng `imgsz=960` để giữ chi tiết quả nhỏ. Checkpoint được ghi trực tiếp lên Drive. Chạy lại cell này sau khi Colab ngắt sẽ resume từ `last.pt`.


In [ ]:
RUNS = '/content/drive/MyDrive/fruit_two_stage/runs'
subprocess.run([sys.executable, f'{REPO}/training/train_two_stage.py',
    '--dataset-root', str(DATASET_ROOT), '--project', RUNS, '--resume',
    '--skip-mango-classifier', '--skip-dragon-classifier'], check=True)


## STEP 6B — Train/resume mango classifier
Bốn lớp xoài được giữ nguyên. Kết quả test sẽ cho biết có đủ bằng chứng để giữ cả bốn hay cần gộp lớp.


In [ ]:
subprocess.run([sys.executable, f'{REPO}/training/train_two_stage.py',
    '--dataset-root', str(DATASET_ROOT), '--project', RUNS, '--resume',
    '--skip-detector', '--skip-dragon-classifier'], check=True)


## STEP 6C — Train/resume dragon-fruit classifier và calibration
Ảnh Mendeley chỉ bổ sung `unripe/ripe`; lớp `rotten` vẫn lấy từ bộ Roboflow. `fresh/defect` không bị ép sai thành `ripe/rotten`.


In [ ]:
subprocess.run([sys.executable, f'{REPO}/training/train_two_stage.py',
    '--dataset-root', str(DATASET_ROOT), '--project', RUNS, '--resume',
    '--skip-detector', '--skip-mango-classifier'], check=True)


In [ ]:
# STEP 7 — Final artifacts and calibrated confidence thresholds
for relative in [
    'detector/weights/best.pt',
    'mango_classifier/weights/best.pt',
    'mango_classifier/calibration.json',
    'dragon_classifier/weights/best.pt',
    'dragon_classifier/calibration.json',
]:
    path = Path(RUNS) / relative
    print(('OK  ' if path.is_file() else 'MISS'), path)
    if path.name == 'calibration.json' and path.is_file():
        print(path.read_text())
